In [ ]:
# Note: 
    # Change hopper file paths to your own in four locations
    # Use trimmed sentinel-2 image (will upload a folder of them to Teams files)
    # Use QGIS to view .tif file at the end. Can change to white background/black pixels by right-clicking layer --> properties --> symbology

In [ ]:
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from sklearn.cluster import MeanShift, estimate_bandwidth

In [ ]:
# Load in sentinel-2 image
image_path = "/home/mlim8/trimmed_S2A_17TLJ_20190714_0_L2A_TCI.tif"

with rasterio.open(image_path) as src:
    image = src.read()
    height, width = image.shape[1], image.shape[2]  # Get full image dimensions

print(f"Full image shape: {image.shape} (Bands, Height, Width)")

In [ ]:
# Compute NDVI (using nir and red bands)
nir_band = image[2].astype(np.float32)  # NIR band
red_band = image[0].astype(np.float32)  # Red band

ndvi = (nir_band - red_band) / (nir_band + red_band + 1e-10)  # Avoid division by zero

In [ ]:
# Save NDVI visualization png
plt.figure(figsize=(8, 8))
plt.imshow(ndvi, cmap='RdYlGn', vmin=-1, vmax=1)
plt.colorbar(label="NDVI Value")
plt.title("NDVI Map - Huron County")
plt.axis("off")

ndvi_output_path = "/home/mlim8/huron_ndviMap_hopper.png"
plt.savefig(ndvi_output_path, dpi=300, bbox_inches="tight")
plt.close()

print(f"NDVI visualization saved at: {ndvi_output_path}")

In [ ]:
# Filter out non-crop areas (where NDVI < 0.4)
valid_pixels = ndvi[ndvi > 0.4].reshape(-1, 1) 

print(f"Total valid crop pixels: {valid_pixels.shape[0]}")

In [ ]:
# Estimate bandwidth for mean shift (determines search radius for grouping points into clusters)
bandwidth = estimate_bandwidth(valid_pixels, quantile=0.3, n_samples=500)
print(f"Estimated bandwidth: {bandwidth}")

In [ ]:
# Apply mean shift clustering
mean_shift = MeanShift(bandwidth=bandwidth, bin_seeding=True)
mean_shift.fit(valid_pixels)

In [ ]:
# Get cluster labels
labels = mean_shift.labels_

In [ ]:
# Reshape clustering results back to full Image
segmented_fields = np.full(ndvi.shape, -1)  # Initialize empty field map
segmented_fields[ndvi > 0.4] = labels  # Assign cluster labels only to valid pixels

print(f"Segmented field shape: {segmented_fields.shape}")

In [ ]:
# Save segmented fields as GeoTIFF
output_path = "/home/mlim8/huron_segmented_hopper.tif"

with rasterio.open(
    output_path,
    "w",
    driver="GTiff",
    height=segmented_fields.shape[0],
    width=segmented_fields.shape[1],
    count=1,
    dtype=labels.dtype,
    crs=src.crs,  # Maintain CRS
    transform=src.transform  # Keep original spatial reference
) as dst:
    dst.write(segmented_fields, 1)

print(f"Segmented full Huron County image saved at: {output_path}")

In [ ]:
# Visualize final segmentation map (colored clusters)
plt.figure(figsize=(8, 8))
plt.imshow(segmented_fields, cmap="tab10")
plt.title("Mean Shift Segmentation - Huron County")
plt.axis("off")
plt.colorbar(label="Cluster Labels")

segmentation_output_path = "/home/mlim8/huron_segmented_map_hopper.png"
plt.savefig(segmentation_output_path, dpi=300, bbox_inches="tight")
plt.close()

print(f"Segmentation map saved at: {segmentation_output_path}")